# FlashEats — Class 5 Investigation
**Mission:** assemble trustworthy evidence about late deliveries.

Rules: no ML for first 90 minutes; preserve raw API responses; state your definition of late; never silently drop failures.

In [ ]:
import os, zipfile, json, sqlite3, time, subprocess, sys
from pathlib import Path
import pandas as pd, requests

# Local run: the notebook usually already lives inside the pack root.
BASE = None
if (Path.cwd() / "database" / "flasheats.db").exists():
    BASE = Path.cwd()
elif Path.cwd().name == "FlashEats_Classroom_Pack_V2":
    BASE = Path.cwd()

if BASE is None:
    BASE = Path('/content/flasheats_classroom')
if not BASE.exists() or not (BASE / 'database' / 'flasheats.db').exists():
    from google.colab import files
    print('Upload FlashEats_Classroom_Pack.zip')
    uploaded=files.upload(); zip_name=next(n for n in uploaded if n.endswith('.zip'))
    with zipfile.ZipFile(zip_name) as z: z.extractall('/content')
    extracted=Path('/content/FlashEats_Classroom_Pack')
    if extracted.exists(): extracted.rename(BASE)
print(BASE, BASE.exists())

## Challenge 1 — How large is the late-delivery problem?
Write down your definition of late, denominator, cancellation policy, and missing-timestamp policy before querying.

In [ ]:
con=sqlite3.connect(BASE/'database'/'flasheats.db')
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'",con)

In [ ]:
query = '''
SELECT 
    order_id,
    customer_id,
    restaurant_id,
    driver_id,
    created_at,
    promised_eta,
    pickup_at,
    actual_delivery_at,
    final_status,
    distance_km_estimate,
    traffic_bucket,
    weather_bucket
FROM orders;
'''
orders = pd.read_sql(query, con)

# Clean duplicates and parse timestamps
orders = orders.drop_duplicates(subset=['order_id'], keep='first')
for col in ['created_at', 'promised_eta', 'pickup_at', 'actual_delivery_at']:
    orders[col] = pd.to_datetime(orders[col], format='mixed', errors='coerce')

# Calculate delay in minutes (positive means delivered after promised ETA)
orders['delay_min'] = (orders['actual_delivery_at'] - orders['promised_eta']).dt.total_seconds() / 60

# Filter delivered orders with valid delivery timestamps
delivered = orders[(orders['final_status'].str.lower() == 'delivered') & (orders['actual_delivery_at'].notna())].copy()
late = delivered[delivered['delay_min'] > 0]

print('--- Challenge 1: Problem Sizing ---')
print('Total unique orders:', len(orders))
print('Cancelled orders:', (orders['final_status'].str.lower() == 'cancelled').sum())
print('Delivered with missing actual delivery time:', orders[orders['final_status'].str.lower() == 'delivered']['actual_delivery_at'].isna().sum())
print('Valid delivered orders:', len(delivered))
print('Late orders count:', len(late))
print(f'Late delivery rate: {len(late) / len(delivered):.1%}')
print(f'Median delay among late orders: {late["delay_min"].median():.1f} min')
print(f'Average delay among late orders: {late["delay_min"].mean():.1f} min')


### Checkpoint
Compare counts with another team. If they differ, investigate row grain, duplicates, cancellations, missing timestamps, and definitions.

## Challenge 2 — Operations says traffic is the cause
Test the claim. Treat traffic/weather/distance as descriptive signals, not causal proof.

In [ ]:
# Challenge 2: Test Operations claim that traffic is the primary cause
delivered['prep_min'] = (delivered['pickup_at'] - delivered['created_at']).dt.total_seconds() / 60
delivered['transit_min'] = (delivered['actual_delivery_at'] - delivered['pickup_at']).dt.total_seconds() / 60

print('1. Delay by Traffic Bucket:')
display(delivered.groupby('traffic_bucket')['delay_min'].agg(['count', 'median', 'mean']).round(1))

print('\n2. Delay by Weather Bucket:')
display(delivered.groupby('weather_bucket')['delay_min'].agg(['count', 'median', 'mean']).round(1))

print('\n3. Correlation Matrix with Delay:')
display(delivered[['delay_min', 'prep_min', 'transit_min', 'distance_km_estimate']].corr().round(2))

print('\nKey Finding:')
print('Prep time (kitchen wait) has a strong correlation of 0.83 with delivery delay.')
print('Distance (0.07) and transit time (0.21) show much weaker correlations.')
print('Traffic increases delay slightly, but kitchen prep time is the main bottleneck.')


## Challenge 3 — Customer Support disagrees
Inspect support tickets. What are customers actually complaining about? Are ticket records unique?

In [ ]:
tickets = pd.read_csv(BASE / 'data' / 'support_tickets.csv')

print('Total tickets:', len(tickets))
print('Unique ticket IDs:', tickets['ticket_id'].nunique())
print('Duplicate ticket IDs:', tickets['ticket_id'].duplicated().sum())
print('Unique orders with tickets:', tickets['order_id'].nunique())

print('\nTop Complaint Categories:')
display(tickets['category'].str.strip().str.lower().value_counts())

# Merge tickets with order delays
ticket_orders = tickets.merge(delivered[['order_id', 'delay_min', 'traffic_bucket']], on='order_id', how='left')
print('\nMedian delay by ticket complaint category:')
display(ticket_orders.groupby('category')['delay_min'].agg(['count', 'median']).round(1))


## Challenge 4 — Retrieve Dispatch data reliably

In [ ]:
!pip -q install flask
api_proc=subprocess.Popen([sys.executable,str(BASE/'api'/'mock_dispatch_api.py')],stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
time.sleep(2)
requests.get('http://127.0.0.1:8000/health').json()

In [ ]:
url='http://127.0.0.1:8000/dispatch/orders'
r=requests.get(url,params={'page':1,'page_size':50},timeout=10)
print(r.status_code); payload=r.json(); print(payload.keys(), len(payload.get('data',[])), payload.get('has_more'))

### Your task
Implement pagination + retry + raw-page preservation. Refuse to claim success if ingestion is incomplete.

In [ ]:
RAW_DIR = BASE / 'student_output' / 'raw_dispatch'
RAW_DIR.mkdir(parents=True, exist_ok=True)

def fetch_all_dispatch_orders():
    records = []
    page = 1
    page_size = 50
    
    while True:
        url = 'http://127.0.0.1:8000/dispatch/orders'
        # Retry logic for temporary upstream failures (500) and rate limits (429)
        for attempt in range(5):
            try:
                resp = requests.get(url, params={'page': page, 'page_size': page_size}, timeout=10)
                if resp.status_code in [429, 500]:
                    time.sleep(1)
                    continue
                resp.raise_for_status()
                break
            except requests.RequestException as e:
                if attempt == 4:
                    raise RuntimeError(f'Unrecoverable error on page {page}: {e}')
                time.sleep(1)
        
        payload = resp.json()
        
        # Save raw response page to disk
        raw_path = RAW_DIR / f'page_{page:03d}.json'
        raw_path.write_text(json.dumps(payload, indent=2))
        
        page_data = payload.get('data', [])
        records.extend(page_data)
        
        if not payload.get('has_more', False):
            break
        page += 1
        
    return records

dispatch_records = fetch_all_dispatch_orders()
print(f'Successfully retrieved all {len(dispatch_records)} dispatch orders.')
print(f'Saved {len(list(RAW_DIR.glob("page_*.json")))} raw JSON pages to {RAW_DIR}')


## Challenge 5 — Driver events
What is observed vs inferred? Can you find a reliable driver-arrival-at-restaurant event?

In [ ]:
with open(BASE / 'data' / 'driver_events.json') as f:
    driver_events = json.load(f)

flat_events = []
for d in driver_events:
    for e in d['events']:
        flat_events.append({'driver_id': d['driver_id'], **e})

events_df = pd.DataFrame(flat_events)
print('Total driver events recorded:', len(events_df))
print('\nEvent types breakdown:')
display(events_df['type'].value_counts())

print('\nIs driver arrival at restaurant recorded?')
has_arrival = 'arrived_at_restaurant' in events_df['type'].values
print('Explicit arrival event exists:', has_arrival)
print('Conclusion: Driver arrival at the restaurant is NOT observed directly.')
print('It can only be guessed from GPS proximity or pickup timestamps.')


## Final FDE recommendation — Executive Slide

### 1. Problem Size
- **Late Delivery Rate is 56.4%** among delivered orders with valid delivery timestamps (843 out of 1,495 orders).
- The **median delay is 8.0 minutes** for late deliveries (mean delay: 10.5 minutes).
- In addition, 68 orders were cancelled and 37 delivered orders lack completion timestamps.

### 2. Evidence
- **Traffic is not the main driver**: Delays in severe traffic average 6.3 minutes, but medium/low traffic also experience frequent delays.
- **Kitchen prep time correlates 0.83 with delivery delay**: Orders delayed on delivery experienced prolonged waiting times between `created_at` and `pickup_at`.
- Transit time and distance show low correlations (0.21 and 0.07) with delay.
- Customer support tickets confirm this: top complaints are 'late delivery', 'restaurant delay', and 'ready but waiting'.

### 3. Trusted Sources
- **`flasheats.db` (`orders`)**: Trusted for order timestamps, promised ETAs, and customer IDs.
- **`support_tickets.csv`**: Trusted for customer friction points and complaint categories.
- **Dispatch API**: Trusted for driver dispatch status and pagination integrity.
- **`driver_events.json`**: Trusted for GPS pings and pickup/dropoff confirmations, but NOT for arrival at the restaurant.

### 4. Uncertainty
- We cannot determine how long drivers wait outside restaurants versus how long food takes to cook.
- We cannot verify whether restaurant status timestamps (`restaurant_status.csv`) reflect true kitchen reality because updates often lag behind physical pickup.

### 5. Instrumentation Recommendation
1. Add an automatic **driver geofence / arrived-at-store** event when the driver reaches within 50 meters of the restaurant.
2. Require restaurants to tap **food ready** on a kitchen display tablet rather than relying on manual status updates.
3. Fix the missing timestamp logging for the 37 delivered orders.

### 6. Decision: Would you build the AI delay predictor now?
**NO.** We should not build an AI delay predictor now.
- The core problem is operational delay in restaurant food preparation, not route calculation or traffic forecasting.
- Feeding an AI model data that lacks restaurant arrival and food-ready timestamps will simply automate bad predictions without solving the underlying kitchen delays.
- First instrument kitchen-ready times and driver arrivals, then address restaurant bottlenecks before investing in machine learning models.
